In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Sentiment_classification

/content/drive/MyDrive/Sentiment_classification


Load processed IMDB Movie Review data

In [3]:
import importlib
from src import preprocess
importlib.reload(preprocess)
from src.preprocess import load_and_preprocess

texts , labels = load_and_preprocess(
    df_path = "/content/drive/MyDrive/Sentiment_classification/data/IMDB Dataset.csv",
    text_column = "review",
    label_column = "sentiment",
    mode = "tfidf"
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
labels

array([1, 1, 1, ..., 0, 0, 0])

In [ ]:
texts.head()

Convert word to vector using TFIDF

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(texts, labels ,test_size = 0.2, random_state = 42)

In [ ]:
from src import vectorizers
importlib.reload(vectorizers)

from src.vectorizers import get_tfidf_vectorizer

vectorizer = get_tfidf_vectorizer(max_features = 10000,ngram_range = (1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)



In [ ]:
X_train_tfidf

In [ ]:
X_test_tfidf

TFIDF with linear models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

models = {
    "Logistic Regression" :LogisticRegression(),
    "Linear Support Vector Classifier": LinearSVC(),
    "Naive Bayes": GaussianNB(),
    "Light GBM": LGBMClassifier()
}

Train and Predict

In [ ]:
from sklearn.metrics import accuracy_score
import time
import pandas as pd

results = []

for name, model in models.items():
  print(f"\n Training {name}")
  start = time.time()
  #naive bayes requires dense matrix
  if name == "Naive Bayes":
    model.fit(X_train_tfidf.toarray(), y_train)
    y_pred = model.predict(X_test_tfidf.toarray())
  else:
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

  end = time.time()
  acc = accuracy_score(y_pred, y_test)

  results.append({
      "Model":name,
      "Max_features":vectorizer.max_features,
      "ngram_range":vectorizer.ngram_range,
      "Accuracy":acc *100,
      "Time":end - start
  })

df_results = pd.DataFrame(results)
df_results


RNN on tfidf data

In [ ]:
import torch
#convert train data to tensor
X_train_tfidf_tensor = torch.tensor(X_train_tfidf.toarray(), dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32)

#convert test data to tensor
X_test_tfidf_tensor = torch.tensor(X_test_tfidf.toarray(), dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.float32)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

#tensors to tensordatasets
X_trainset = TensorDataset(X_train_tfidf_tensor, y_train_tensor)
X_testset = TensorDataset(X_test_tfidf_tensor, y_test_tensor)

#Dataloaders
train_loader = DataLoader(X_trainset, batch_size = 64, shuffle = True)
test_loader = DataLoader(X_testset, batch_size = 64)

In [ ]:
#RNN layer
import torch.nn as nn
import torch.optim as optim
class SentimentRNN(nn.Module):
  def __init__(self, input_size, hidden_size = 128,num_layers = 1):
    super().__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers,
                      batch_first = True)
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):

    outputs, _ = self.rnn(x)

    out = self.fc(outputs[:,-1,:])
    return out



In [ ]:
sentiment_rnn = SentimentRNN(input_size = X_train_tfidf.shape[1])

criterion = nn.BCELoss()
optimizer = optim.Adam(sentiment_rnn.parameters())

In [ ]:
#Train
epochs = 15
rnn_start = time.time()
for epoch in range(epochs):
  sentiment_rnn.train()
  running_loss = 0.0
  for xb, yb in train_loader:
    #set gradients to zero
    optimizer.zero_grad()
    #xb shape -> [batch_size, input_size]
    #model expects -> [batch_size, sequence_length, input_size]
    #add singlton value
    xb = xb.unsqueeze(1)
    #predict
    output = sentiment_rnn.forward(xb)
    #output shape -> [batch_size, 1] and bceloss expects values between 0 and 1
    #squeeze output ->[batch_size] and apply sigmoid to convert values to 0 and 1
    output = torch.sigmoid(output.squeeze())

    #calculate loss
    loss = criterion(output, yb)

    #calculate gradients through backward propagation
    loss.backward()

    #update gradients
    optimizer.step()

    running_loss+= loss.item()

  #Average train loss per epoch
  print(f"{epoch +1}/{epochs} and loss {running_loss/len(train_loader)}")

rnn_end = time.time()

In [ ]:
sentiment_rnn.eval()
with torch.no_grad():
  total_values = 0.0
  correct_values = 0.0

  for xb, yb in test_loader:
    xb = xb.unsqueeze(1)
    output = sentiment_rnn.forward(xb)
    #split probabilities based on threshold 0.5
    predicted = (torch.sigmoid(output.squeeze())>0.5).float()
    correct_values +=(predicted == yb).sum().item()
    total_values += yb.size(0)
  acc = correct_values/total_values*100
  print(f"Accuracy: {acc}")



In [ ]:
rnn_result = pd.DataFrame([{
    "Model": "RNN(relu)",
    "Max_features":vectorizer.max_features,
    "ngram_range":vectorizer.ngram_range,
    "Accuracy": acc,
    "Time":rnn_end - rnn_start
}])

In [ ]:
!git add .

In [ ]:
!git commit -m "cleaned notebook outputs"

[main 53f0f10] cleaned notebook outputs
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
!git push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 483 bytes | 23.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Sravani-AIML/Sentiment_classification.git
   0d23f65..53f0f10  main -> main
